# 01 — Computational Graphs & Autograd from Scratch

**Goal:** Build a complete autograd engine that supports neural network operations.  
**Core insight:** Reverse-mode autodiff (backpropagation) computes gradients of **one scalar output** w.r.t. **all inputs** in a single backward pass — O(1) in number of outputs.

---

## Key Formulas

**Chain rule (scalar):** If $y = f(g(x))$, then $\frac{dy}{dx} = \frac{dy}{dg} \cdot \frac{dg}{dx}$

**Chain rule (multivariate):** If $L = L(y_1, \ldots, y_m)$ and each $y_j = y_j(x_1, \ldots, x_n)$:
$$\frac{\partial L}{\partial x_i} = \sum_j \frac{\partial L}{\partial y_j} \cdot \frac{\partial y_j}{\partial x_i}$$

**Reverse mode:** Propagate $\bar{x}_i = \frac{\partial L}{\partial x_i}$ (adjoint) from output to inputs.  
**Forward mode:** Propagate $\dot{y}_j = \frac{\partial y_j}{\partial x_i}$ (tangent) from one input to all outputs.

| Mode | Cost per pass | Best when |
|------|--------------|----------|
| Reverse (backprop) | O(1) backward passes for 1 scalar output | #outputs << #inputs (neural nets) |
| Forward | O(1) forward passes per input | #inputs << #outputs (rare in DL) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(42)
%matplotlib inline

## 1. The `Value` Class — Scalar Autograd Engine

Each `Value` wraps a scalar and tracks:
- `data`: the numerical value
- `grad`: accumulated gradient (adjoint) $\partial L / \partial \text{self}$
- `_backward`: a closure that propagates gradient to children
- `_prev`: set of parent nodes for topological sort

In [ ]:
class Value:
    """Scalar-valued autograd node."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"
    
    # ---- Arithmetic ops ----
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad    # d(a+b)/da = 1
            other.grad += 1.0 * out.grad   # d(a+b)/db = 1
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return (-self) + other
    
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad   # d(a*b)/da = b
            other.grad += self.data * out.grad   # d(a*b)/db = a
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __truediv__(self, other):
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return other * (self ** -1)
    
    def __pow__(self, n):
        assert isinstance(n, (int, float)), "only int/float powers"
        out = Value(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out
    
    # ---- Transcendental ops ----
    def exp(self):
        out = Value(np.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad   # d(exp(x))/dx = exp(x)
        out._backward = _backward
        return out
    
    def log(self):
        out = Value(np.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out
    
    # ---- Activation helpers ----
    def relu(self):
        out = Value(max(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (1.0 if self.data > 0 else 0.0) * out.grad
        out._backward = _backward
        return out
    
    def sigmoid(self):
        s = 1.0 / (1.0 + np.exp(-self.data))
        out = Value(s, (self,), 'sigmoid')
        def _backward():
            self.grad += s * (1 - s) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        t = np.tanh(self.data)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1.0 - t**2) * out.grad
        out._backward = _backward
        return out
    
    # ---- Backward pass ----
    def backward(self):
        """Reverse-mode autodiff from this node."""
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0  # dL/dL = 1
        for node in reversed(topo):
            node._backward()

print("Value class defined.")

## 2. Topological Sort — Why It Matters

The backward pass must process nodes in **reverse topological order** — each node's gradient must be fully accumulated before propagating further. This is a DFS-based sort.

**Gradient accumulation for shared nodes:** When a node feeds into multiple operations, its gradient is the **sum** of all incoming gradients (multivariate chain rule).

In [ ]:
# Demo: shared node — gradient accumulation
a = Value(3.0, label='a')
b = a + a  # 'a' is used twice => grad should be 2.0
b.backward()
print(f"a = {a.data}, b = a + a = {b.data}")
print(f"db/da = {a.grad}  (should be 2.0 — gradient accumulated from both paths)")

# More complex: a feeds into two different ops
a = Value(2.0, label='a')
b = a * 3        # db/da = 3
c = a ** 2        # dc/da = 2a = 4
d = b + c         # dd/da = db/da + dc/da = 3 + 4 = 7
d.backward()
print(f"\na=2, d = 3a + a^2 = {d.data}")
print(f"dd/da = {a.grad}  (should be 3 + 2*2 = 7)")

In [ ]:
# Visualize the topological order
def trace(root):
    """Build sets of all nodes and edges in the graph."""
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def topo_sort(root):
    topo = []
    visited = set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)
            topo.append(v)
    build(root)
    return topo

# Build a small graph
x = Value(1.0, label='x')
w = Value(-3.0, label='w')
b_val = Value(6.0, label='b')
y = (x * w + b_val).tanh()
y.label = 'y'

order = topo_sort(y)
print("Topological order (forward):")
for i, node in enumerate(order):
    print(f"  {i}: {node.label or node._op} = {node.data:.4f}")

print("\nBackward pass processes in REVERSE of this order.")

## 3. Verification — Numerical Gradient Checking

$$\frac{\partial f}{\partial x} \approx \frac{f(x + \epsilon) - f(x - \epsilon)}{2\epsilon}$$

This is the gold standard for verifying backprop correctness. Use $\epsilon \approx 10^{-5}$.

In [ ]:
def numerical_grad(f, x_val, eps=1e-5):
    """Compute df/dx numerically."""
    return (f(x_val + eps) - f(x_val - eps)) / (2 * eps)

# Test all operations
tests = [
    ("add",     lambda xv: (Value(xv) + Value(2.0)).data,         lambda xv: Value(xv) + Value(2.0)),
    ("mul",     lambda xv: (Value(xv) * Value(3.0)).data,         lambda xv: Value(xv) * Value(3.0)),
    ("pow",     lambda xv: (Value(xv) ** 3).data,                 lambda xv: Value(xv) ** 3),
    ("exp",     lambda xv: Value(xv).exp().data,                  lambda xv: Value(xv).exp()),
    ("log",     lambda xv: Value(xv).log().data,                  lambda xv: Value(xv).log()),
    ("relu",    lambda xv: Value(xv).relu().data,                 lambda xv: Value(xv).relu()),
    ("sigmoid", lambda xv: Value(xv).sigmoid().data,              lambda xv: Value(xv).sigmoid()),
    ("tanh",    lambda xv: Value(xv).tanh().data,                 lambda xv: Value(xv).tanh()),
]

x_test = 1.5
print(f"{'Op':<10} {'Analytical':>12} {'Numerical':>12} {'Match':>8}")
print("-" * 45)
for name, f_scalar, f_value in tests:
    # Numerical
    ng = numerical_grad(f_scalar, x_test)
    # Analytical
    x = Value(x_test)
    out = f_value(x_test)  # This creates a new graph
    # Need to rebuild for analytical
    x = Value(x_test)
    if name == 'add': out = x + Value(2.0)
    elif name == 'mul': out = x * Value(3.0)
    elif name == 'pow': out = x ** 3
    elif name == 'exp': out = x.exp()
    elif name == 'log': out = x.log()
    elif name == 'relu': out = x.relu()
    elif name == 'sigmoid': out = x.sigmoid()
    elif name == 'tanh': out = x.tanh()
    out.backward()
    ag = x.grad
    match = abs(ag - ng) < 1e-4
    print(f"{name:<10} {ag:>12.6f} {ng:>12.6f} {'OK' if match else 'FAIL':>8}")

## 4. The `Tensor` Class — Array-valued Autograd

For neural networks we need array operations: matmul, element-wise ops on arrays, reductions (sum). This extends our scalar engine to NumPy arrays.

In [ ]:
class Tensor:
    """Array-valued autograd node backed by NumPy."""
    
    def __init__(self, data, _children=(), _op='', label=''):
        self.data = np.array(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data, dtype=np.float64)
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
        self.label = label
    
    @property
    def shape(self):
        return self.data.shape
    
    def __repr__(self):
        return f"Tensor(shape={self.shape}, data={self.data})"
    
    # ---- Element-wise ops ----
    def __add__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data + other.data, (self, other), '+')
        def _backward():
            # Handle broadcasting: sum over broadcasted dims
            g = out.grad
            self.grad += _unbroadcast(g, self.shape)
            other.grad += _unbroadcast(g, other.shape)
        out._backward = _backward
        return out
    
    def __radd__(self, other):
        return self + other
    
    def __neg__(self):
        return self * (-1.0)
    
    def __sub__(self, other):
        return self + (-other)
    
    def __rsub__(self, other):
        return (-self) + other
    
    def __mul__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += _unbroadcast(other.data * out.grad, self.shape)
            other.grad += _unbroadcast(self.data * out.grad, other.shape)
        out._backward = _backward
        return out
    
    def __rmul__(self, other):
        return self * other
    
    def __truediv__(self, other):
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data / other.data, (self, other), '/')
        def _backward():
            self.grad += _unbroadcast(out.grad / other.data, self.shape)
            other.grad += _unbroadcast(-self.data / (other.data ** 2) * out.grad, other.shape)
        out._backward = _backward
        return out
    
    def __pow__(self, n):
        assert isinstance(n, (int, float))
        out = Tensor(self.data ** n, (self,), f'**{n}')
        def _backward():
            self.grad += n * (self.data ** (n - 1)) * out.grad
        out._backward = _backward
        return out
    
    # ---- Transcendental ----
    def exp(self):
        out = Tensor(np.exp(self.data), (self,), 'exp')
        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    
    def log(self):
        out = Tensor(np.log(self.data), (self,), 'log')
        def _backward():
            self.grad += (1.0 / self.data) * out.grad
        out._backward = _backward
        return out
    
    # ---- Reductions ----
    def sum(self, axis=None, keepdims=False):
        out = Tensor(np.sum(self.data, axis=axis, keepdims=keepdims), (self,), 'sum')
        def _backward():
            g = out.grad
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis=axis)
            self.grad += np.broadcast_to(g, self.shape)
        out._backward = _backward
        return out
    
    def mean(self, axis=None, keepdims=False):
        n = self.data.size if axis is None else self.data.shape[axis]
        return self.sum(axis=axis, keepdims=keepdims) * (1.0 / n)
    
    # ---- Matmul ----
    def matmul(self, other):
        """Matrix multiply: self @ other."""
        other = other if isinstance(other, Tensor) else Tensor(other)
        out = Tensor(self.data @ other.data, (self, other), '@')
        def _backward():
            # For 2D: dL/dA = dL/dC @ B^T, dL/dB = A^T @ dL/dC
            self.grad += out.grad @ other.data.T
            other.grad += self.data.T @ out.grad
        out._backward = _backward
        return out
    
    def __matmul__(self, other):
        return self.matmul(other)
    
    # ---- Activations ----
    def relu(self):
        out = Tensor(np.maximum(0, self.data), (self,), 'relu')
        def _backward():
            self.grad += (self.data > 0).astype(float) * out.grad
        out._backward = _backward
        return out
    
    def sigmoid(self):
        s = 1.0 / (1.0 + np.exp(-self.data))
        out = Tensor(s, (self,), 'sigmoid')
        def _backward():
            self.grad += s * (1 - s) * out.grad
        out._backward = _backward
        return out
    
    def tanh(self):
        t = np.tanh(self.data)
        out = Tensor(t, (self,), 'tanh')
        def _backward():
            self.grad += (1.0 - t**2) * out.grad
        out._backward = _backward
        return out
    
    # ---- Backward ----
    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.grad = np.ones_like(self.data)
        for node in reversed(topo):
            node._backward()


def _unbroadcast(grad, target_shape):
    """Sum out dimensions that were broadcast."""
    # Pad target_shape on the left to match grad.ndim
    while len(target_shape) < grad.ndim:
        grad = grad.sum(axis=0)
    for i, (gs, ts) in enumerate(zip(grad.shape, target_shape)):
        if ts == 1 and gs != 1:
            grad = grad.sum(axis=i, keepdims=True)
    return grad


print("Tensor class defined.")

In [ ]:
# Quick test: matmul gradient
A = Tensor(np.random.randn(2, 3))
B = Tensor(np.random.randn(3, 4))
C = A @ B
loss = C.sum()
loss.backward()

# Numerical check for A[0,0]
eps = 1e-5
A_plus = A.data.copy(); A_plus[0, 0] += eps
A_minus = A.data.copy(); A_minus[0, 0] -= eps
num_grad = (np.sum(A_plus @ B.data) - np.sum(A_minus @ B.data)) / (2 * eps)

print(f"dL/dA[0,0] analytical: {A.grad[0,0]:.6f}")
print(f"dL/dA[0,0] numerical:  {num_grad:.6f}")
print(f"Match: {abs(A.grad[0,0] - num_grad) < 1e-4}")

## 5. Demo — Gradients for a Neural Network Forward Pass

Build a 2-layer neural net: $\hat{y} = \sigma(W_2 \cdot \text{ReLU}(W_1 x + b_1) + b_2)$

Compute MSE loss and get gradients w.r.t. all parameters.

In [ ]:
# A 2-layer network: 2 -> 4 -> 1
np.random.seed(42)

# Parameters
W1 = Tensor(np.random.randn(2, 4) * 0.5, label='W1')
b1 = Tensor(np.zeros((1, 4)), label='b1')
W2 = Tensor(np.random.randn(4, 1) * 0.5, label='W2')
b2 = Tensor(np.zeros((1, 1)), label='b2')

# Input and target
x = Tensor(np.array([[1.0, 2.0]]), label='x')   # (1, 2)
y_true = Tensor(np.array([[1.0]]), label='y_true')

# Forward pass
h = (x @ W1 + b1).relu()   # hidden layer with ReLU
y_pred = (h @ W2 + b2).sigmoid()  # output with sigmoid

# MSE loss
diff = y_pred - y_true
loss = (diff * diff).mean()

# Backward
loss.backward()

print(f"Forward pass:")
print(f"  Hidden: {h.data}")
print(f"  Prediction: {y_pred.data[0,0]:.4f}")
print(f"  Loss: {loss.data:.4f}")
print(f"\nGradients:")
print(f"  dL/dW1 shape: {W1.grad.shape}")
print(f"  dL/dW1:\n{W1.grad}")
print(f"  dL/db1: {b1.grad}")
print(f"  dL/dW2:\n{W2.grad}")
print(f"  dL/db2: {b2.grad}")

In [ ]:
# Numerical gradient check for W1[0,0]
eps = 1e-5

def forward_pass(W1_data, W2_data, b1_data, b2_data, x_data, y_data):
    h = np.maximum(0, x_data @ W1_data + b1_data)
    s = 1.0 / (1.0 + np.exp(-(h @ W2_data + b2_data)))
    return np.mean((s - y_data) ** 2)

W1_p = W1.data.copy(); W1_p[0, 0] += eps
W1_m = W1.data.copy(); W1_m[0, 0] -= eps
num = (forward_pass(W1_p, W2.data, b1.data, b2.data, x.data, y_true.data) -
       forward_pass(W1_m, W2.data, b1.data, b2.data, x.data, y_true.data)) / (2 * eps)

print(f"dL/dW1[0,0]:")
print(f"  Autograd:  {W1.grad[0,0]:.8f}")
print(f"  Numerical: {num:.8f}")
print(f"  Abs diff:  {abs(W1.grad[0,0] - num):.2e}")

## 6. Reverse Mode vs Forward Mode — Intuition

**Reverse mode (backprop):**
- One backward pass gives $\frac{\partial L}{\partial x_i}$ for ALL parameters $x_i$
- Cost: one forward + one backward pass (roughly 2x forward)
- Neural nets have 1 scalar loss, millions of parameters => perfect fit

**Forward mode:**
- One forward pass gives $\frac{\partial y_j}{\partial x_i}$ for ALL outputs $y_j$ but only ONE input $x_i$
- Need N forward passes for N parameters
- Useful when #inputs < #outputs (e.g., computing Jacobian columns)

For a network with $P$ parameters and 1 loss:
- Reverse mode: **1 pass** (what we use)
- Forward mode: **P passes** (infeasible for P = millions)

In [ ]:
# Visualization: cost comparison
params = np.logspace(1, 7, 50)
reverse_cost = np.ones_like(params) * 2  # ~2x forward pass, constant
forward_cost = params  # one pass per parameter

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.loglog(params, reverse_cost, 'b-', linewidth=2, label='Reverse mode (backprop)')
ax.loglog(params, forward_cost, 'r--', linewidth=2, label='Forward mode')
ax.axvline(x=1e6, color='gray', linestyle=':', alpha=0.5)
ax.text(1e6, 1e5, 'Typical NN\n(~1M params)', ha='center', fontsize=9)
ax.set_xlabel('Number of parameters')
ax.set_ylabel('Relative cost (# passes)')
ax.set_title('Why Reverse Mode Wins for Neural Networks')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Training a Tiny Network with Our Autograd

Proof that our engine works: learn XOR.

In [ ]:
# XOR dataset
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=np.float64)
Y_xor = np.array([[0], [1], [1], [0]], dtype=np.float64)

# Initialize parameters
np.random.seed(42)
W1 = Tensor(np.random.randn(2, 8) * 0.5)
b1 = Tensor(np.zeros((1, 8)))
W2 = Tensor(np.random.randn(8, 1) * 0.5)
b2 = Tensor(np.zeros((1, 1)))
params = [W1, b1, W2, b2]

lr = 0.5
losses = []

for epoch in range(500):
    # Zero gradients
    for p in params:
        p.grad = np.zeros_like(p.data)
    
    # Forward
    x = Tensor(X_xor)
    y = Tensor(Y_xor)
    h = (x @ W1 + b1).tanh()
    pred = (h @ W2 + b2).sigmoid()
    
    # MSE loss
    diff = pred - y
    loss = (diff * diff).mean()
    losses.append(loss.data)
    
    # Backward
    loss.backward()
    
    # SGD update
    for p in params:
        p.data -= lr * p.grad

# Results
print("Predictions after training:")
h = Tensor(np.maximum(0, X_xor @ W1.data + b1.data))
final_h = np.tanh(X_xor @ W1.data + b1.data)
final_pred = 1.0 / (1.0 + np.exp(-(final_h @ W2.data + b2.data)))
for i in range(4):
    print(f"  {X_xor[i]} => {final_pred[i,0]:.4f} (target: {Y_xor[i,0]})")

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(losses)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('XOR Training with Our Autograd Engine')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. PyTorch Comparison

Verify our implementation matches PyTorch's autograd.

In [ ]:
try:
    import torch
    
    # Same computation in both
    np.random.seed(0)
    w_np = np.random.randn(3, 2)
    x_np = np.random.randn(1, 3)
    
    # Our engine
    W_ours = Tensor(w_np.copy())
    x_ours = Tensor(x_np.copy())
    out_ours = (x_ours @ W_ours).sum()
    out_ours.backward()
    
    # PyTorch
    W_pt = torch.tensor(w_np.copy(), requires_grad=True)
    x_pt = torch.tensor(x_np.copy(), requires_grad=True)
    out_pt = (x_pt @ W_pt).sum()
    out_pt.backward()
    
    print("dL/dW comparison:")
    print(f"  Ours:    {W_ours.grad}")
    print(f"  PyTorch: {W_pt.grad.numpy()}")
    print(f"  Match: {np.allclose(W_ours.grad, W_pt.grad.numpy())}")
    
    print(f"\ndL/dx comparison:")
    print(f"  Ours:    {x_ours.grad}")
    print(f"  PyTorch: {x_pt.grad.numpy()}")
    print(f"  Match: {np.allclose(x_ours.grad, x_pt.grad.numpy())}")
    
except ImportError:
    print("PyTorch not available — skipping comparison.")

## Interview Quick Reference

**Q: What is a computational graph?**  
A DAG where nodes are operations and edges represent data flow. Enables automatic differentiation by recording the forward computation.

**Q: Why reverse mode for neural nets?**  
Neural nets have one scalar loss but millions of parameters. Reverse mode computes all gradients in one backward pass. Forward mode would need one pass per parameter.

**Q: What is gradient accumulation?**  
When a node is used in multiple downstream operations, its gradient is the sum of gradients flowing back from each use (multivariate chain rule).

**Q: What does `loss.backward()` do in PyTorch?**  
1. Topologically sorts the computation graph
2. Traverses in reverse order
3. At each node, applies local chain rule to compute gradients
4. Accumulates gradients in `.grad` attributes

**Q: Static vs dynamic graphs?**  
- Static (TF1): graph built once, then executed — allows optimization but harder to debug
- Dynamic (PyTorch, TF2 eager): graph rebuilt each forward pass — flexible, debuggable

---
*Next: 02_neural_network_from_scratch.ipynb — building proper layer abstractions*